# Import Packages

In [1]:
## usual impport functions : general use
import numpy as np
from numpy import cosh, zeros_like, mgrid, zeros
import matplotlib.pyplot as plt
import pandas as pd
import math

# from interpolation.complete_poly import (CompletePolynomial,
#                                          n_complete, complete_polynomial,
#                                          complete_polynomial_der,
#                                          _complete_poly_impl,
#                                          _complete_poly_impl_vec,
#                                          _complete_poly_der_impl,
#                                          _complete_poly_der_impl_vec)
from numba import jit, vectorize, njit, prange

## scipy functions for optimization (not all used)
import scipy.optimize as opt
from scipy.optimize import root
from scipy.sparse import spdiags, kron
from scipy.sparse.linalg import spilu, LinearOperator, eigs
from scipy.optimize import leastsq
from scipy.optimize import minimize
from scipy.special import j1
from scipy.sparse import lil_matrix, coo_matrix
from scipy.interpolate import interp1d ## 1d interpolation, 2d and radial basis are avaiable for higher dimension interpolation
from typing import NamedTuple, Callable
from collections import namedtuple
from scipy.special import roots_hermite
from scipy.interpolate import RegularGridInterpolator


from concurrent.futures import ProcessPoolExecutor
from multiprocessing import Pool
import requests
import time
from time import time

from mpl_toolkits.mplot3d import Axes3D


from dataclasses import dataclass
from numpy.linalg import eig

import os
print(f"Available CPUs: {os.cpu_count()}")
### quantecon functions
# import quantecon as qe
# import quantecon_wasm as qe_w
# from quantecon import compute_fixed_point
# from quantecon.markov import DiscreteDP

Available CPUs: 2


# Functions

In [2]:
## rouwenhorst method

## AR process - yt = (1- rho)*y_bar + rho*yt + et, sigma_e

def rouwenhorst_matrix(params):
  p = (1+params.rho)/2
  q = p

  mat_temp=np.array([[p,1-p],[1-q,q]])
  sigma_y = params.sigma / np.sqrt(1 - params.rho**2)
  delta = sigma_y*np.sqrt(params.nbn-1)


  for i in range(params.nbn-2):
    a = mat_temp.shape[0]
    mat_temp_next = np.zeros((a+1,a+1))
    mat_temp_next[:a,:a] += p*mat_temp
    mat_temp_next[:a,1:] += (1-p)*mat_temp
    mat_temp_next[1:,:a] += (1-p)*mat_temp
    mat_temp_next[1:,1:] += p*mat_temp
    mat_temp = mat_temp_next
    mat_temp[1:a,:] = mat_temp[1:a,:]/2

  return mat_temp, delta


def stationary_distribution(Pi):

  vals, vecs = np.linalg.eig(Pi)
  index = np.argmin(np.abs(vals - 1.0))
  pi = vecs[:, index].real
  pi = pi / np.sum(pi)

  return pi


def stationary_distribution_stable(Pi,initial, max_iter=1000,tol = 1e-07):
  pi = initial
  for _ in range(max_iter):
    pi_new = Pi.T @ pi
    if np.max(np.abs(pi_new - pi)) < tol:
        return pi_new
    pi = pi_new

  return pi


def discretize_state(params):
  p = (1+params.rho)/2
  Pi , delta = rouwenhorst_matrix(params)         ## discretize a log normal AR process so exp
  z = np.exp(np.linspace(-delta, delta, params.nbn))
  N = Pi.shape[0]
  initial = np.ones(N) / N  # start uniform
  pi = stationary_distribution_stable(Pi,initial)

  return Pi, pi, z

## functions


def f(params,k,z):
  return z*k**(params.alpha)


def fprime(params,k,z):
  return z*params.alpha*k**(params.alpha-1)


def u_c(params,c):
  if params.gamma == 1:
    return np.log(c)
  else:
    return c**(params.gamma-1)/(params.gamma-1)


def mu_c(params,c):
  return c**(-params.gamma)


def mu_c_inv(params,y):
  return y** (-1/params.gamma)


def chebyshev_grid(k_min, k_max, N):
    x = chebyshev_nodes(N)              # in [-1,1]
    return 0.5*(k_max - k_min)*(x + 1) + k_min


def cheby_pol(kz, order): ## chebyshev polynomial - usual
  N = len(kz)                     ## Chebypol = chebvander(vChebynodes, ChebyOrder) - alternative
  T = np.zeros((N, order+1))
  T[:,0] = 1

  if order >= 1:
    for i in range(N):
      T[i,1] = kz[i]
## cheby recursion
  for n in range(1, order):
    for i in range(N):
      T[i,n+1] = 2*kz[i]*T[i,n] - T[i,n-1]
  return

## functions necessary

def chebyshev_nodes(n):
  return np.cos((2*np.arange(1,n+1)+1)*np.pi/(2*n+2)) ## return n sized chebyshev nodes (n*1)


def chebyshev_approx(kappa, x):
    return chebval(x, kappa)  ### return the chebyshev function numerical value with known cheby coefficients


def chebyshev_coefficients(c_values, x_nodes, order = 3):
    """
    c_values: values of the policy function at Chebyshev nodes
    order: maximum polynomial order N. default is 3
    x_nodes: Chebyshev nodes on [-1,1]
    """
    coeffs = np.zeros(order+1)
    for p in range(order+1):
        Tp = np.cos(p * np.arccos(x_nodes))  # Chebyshev polynomial at nodes
        coeffs[p] = np.sum(c_values * Tp) / np.sum(Tp**2)
    return coeffs

# def combine_cheby(k_cheby,z_cheby):
#   k_num= k_cheby.size
#   z_num= z_cheby.size
#   max_num = max(k_num, z_num)
#   logc = np.zeros(max_num)
#   for i in range(k_num):
#     for j in range(z_num)

# Parameteres

In [ ]:


## namedtuple works with numba - dictionary not compliant with that
Params = namedtuple("Params", ["r", "alpha",
                               "beta",
                               "delta",
                               "gamma",   ## IES
                               "rho",
                               "sigma",
                                "k_min",  # borrowing constraint
                               "nbk",           ## grid size  - needed for time iteration or egm
                                "k_max",
                                "nbn", ## number of states of discretized state space
                               "n_bar",
                               "rou",
                               "B",
                               "Xbar",
                               "sh_rho"
                                  ])

params = Params(
    beta=0.96,
     delta = 1,
    gamma=1,
    alpha=1/3,
    rho=0.9,
    sigma=0.2,
    r=0.03,
    k_min=0,                ## assest/capital in log - lower bound is 0 not inclusive
    k_max=100,
    nbk= 500,
    nbn = 7,
    n_bar = 1,            ## makes it 0 anyways
    rou = 1,
    B = 2,              ## no of bonds

    Xbar = 1,               ## mean productivity
    sh_rho = 0.95         ## AR shock values for MIT shock
)


In [ ]:
1/params.beta - 1

## steady state distribution exists for r only less than this below

0.04166666666666674

# RBC model

In [ ]:

def create_grid(params):
  """
  func : create the state space for the problem
  params : namedtuple with all the parameters
  creates the capital and idiosyncratic labor/income grid
  returns both grid and transistion and stationary distribution for idiosyncratic labor

  """

  ## non linear grid
  # a_uniform = np.linspace(0, 1, params.nbk)
  # a_transformed = a_uniform**2
  # k_grid = params.k_min + (params.k_max - params.k_min) * a_transformed## continous state/ discrete state

  ubar = np.log(1 + np.log(1 + params.k_max - params.k_min))
  a_uniform = np.linspace(0, ubar, params.nbk)
  k_grid = params.k_min + np.exp(np.exp(a_uniform) - 1) - 1

  return k_grid, Pi, pi


def euler_inv_RW_1(params, Pi, c_pol, r):
  """
  function : calculates the inverse of the euler equation
  params : namedtuple with all the parameters
  k_grid : grid for capital used in interpolant - np.array
  c_policy : consumption policy function used in interpolant - np.array

  """
  ## muc
  mu_c_arr = mu_c(params, c_pol) * (1+r)              #### dimesion - (nbk*nbn)
  expec = (Pi @ mu_c_arr.T).T

  return mu_c_inv(params, params.beta * expec)

### egm - 3rd fastest it seems.

## interpolate to get saving polciy as function todays asset - similar to 2
def solver_egm_3(params, r, k_grid, Pi, y_grid, n_grid, tol = 1e-6, error = 1, iter = 0, max_iter = 1000):

  ## starting array for c
  c_old =  0.05*(y_grid[None, :] +  (1+r)*k_grid[:, None]) #### dimesion - (nbk*nbn)

  ## iteration
  for iter in range(max_iter):

    c_new = np.empty_like(c_old)
    k_new = np.empty_like(c_old)

    ## calculate the c - from euler equation
    c_1 = euler_inv_RW_1(params, Pi, c_old, r)              #### dimesion - (nbk*nbn)
    ## calculate the k - from budget equation
    #print(z)
    k_1 = (k_grid[:, None] + c_1 - y_grid[None, :])/(1+r)

    for iy in range(params.nbn):

      k_new[:, iy] = np.interp(k_grid, k_1[:, iy], k_grid)

    # borrowing constraint
    k_new = np.maximum(k_new, k_grid[0])
    c_new = y_grid[None, :] +  (1+r)*k_grid[:, None] - k_new

    error = np.max(np.abs(c_new - c_old))
    c_old[:] = c_new

    if error < tol:
      print(f"Converged in {iter} iterations")
      break
  return c_old, k_grid, k_new

class Baseline_RBC_1:

    def __init__(self, params, Pi, pi, y_grid, n_grid, r):
        """
        class creates a solver for an aiyagari economy using EGM
        """
        self.params = params
        self.r = r
        self.Pi = Pi
        self.pi = pi
        self.y_grid = y_grid


    def EGM_1(self, params, Pi, pi, y_grid, n_grid, r, tol = 10**-6, max_iter = 1000):

      k_grid, Pi, pi = create_grid(params)
      ## return the policy functions through njit class
      return solver_egm_3(params, r, k_grid, Pi, y_grid, n_grid, tol = 1e-6, error = 1, iter = 0, max_iter = 1000), Pi, pi



## Calibration

## Dynamics : Finding equilibrium

### Sequence space jacobian

#### Brute force method

#### Fake news algo